# Adapter: sensores de humedad agricola

## Introduccion

Un sistema de riego espera una interfaz `leer_humedad` que retorne porcentaje. Un sensor externo ya instalado expone `read_moisture` y devuelve una fraccion entre 0 y 1. Adapter permite reutilizar el sensor sin modificar su codigo y traduce tanto el nombre del metodo como la unidad.

## Sin patron

El controlador llama la interfaz que espera, pero el sensor externo tiene otro metodo y otro formato de dato.

In [8]:
# Adaptee: API externa que no podemos modificar.
class SensorExterno:
    def read_moisture(self, plot_id):
        return {"plot": plot_id, "fraction": 0.42}

# El controlador supone que todos los sensores tienen leer_humedad.
def decidir_riego(sensor, plot_id):
    humedad = sensor.leer_humedad(plot_id)
    return "regar" if humedad < 40 else "no regar"

try:
    print(decidir_riego(SensorExterno(), "Lote-7"))
except AttributeError as error:
    print(f"Interfaz incompatible: {error}")

Interfaz incompatible: 'SensorExterno' object has no attribute 'leer_humedad'


## Con Adapter

`SensorHumedad` es la interfaz esperada. `AdaptadorSensorExterno` la implementa, delega al adaptee y convierte la fraccion a porcentaje.

In [10]:
from abc import ABC, abstractmethod

# Target: interfaz que necesita el controlador de riego.
class SensorHumedad(ABC):
    @abstractmethod
    def leer_humedad(self, plot_id):
        raise NotImplementedError

# El adaptee conserva su API original y su unidad en fraccion.
class SensorExterno:
    def read_moisture(self, plot_id):
        return {"plot": plot_id, "fraction": 0.42}

# Adapter: presenta Target y traduce la respuesta del Adaptee.
class AdaptadorSensorExterno(SensorHumedad):
    def __init__(self, sensor):
        self.sensor = sensor

    def leer_humedad(self, plot_id):
        # Ademas de renombrar el metodo, convierte fraccion a porcentaje.
        lectura = self.sensor.read_moisture(plot_id)
        return lectura["fraction"] * 100

# El controlador depende de la interfaz Target, no de la API externa.
class ControladorRiego:
    def decidir_riego(self, sensor: SensorHumedad, plot_id):
        humedad = sensor.leer_humedad(plot_id)
        return f"{plot_id}: {humedad:.0f}% - " + ("regar" if humedad < 40 else "no regar")

# El controlador recibe el adapter como si fuera un SensorHumedad real.
sensor = AdaptadorSensorExterno(SensorExterno())
controlador = ControladorRiego()
print(controlador.decidir_riego(sensor, "Lote-7"))

Lote-7: 42% - no regar


## UML

```plantuml
@startuml
interface SensorHumedad {
  +leer_humedad(plot_id)
}
class AdaptadorSensorExterno {
  -sensor: SensorExterno
  +leer_humedad(plot_id)
}
class SensorExterno {
  +read_moisture(plot_id)
}
class ControladorRiego {
  +decidir_riego(sensor, plot_id)
}
SensorHumedad <|.. AdaptadorSensorExterno
AdaptadorSensorExterno --> SensorExterno : adapta
ControladorRiego --> SensorHumedad : usa
@enduml
```

## Justificacion

Elegi Adapter porque el sensor ya existe y no se puede cambiar, pero su interfaz y unidad no coinciden con el controlador. Factory Method trataria creacion y Strategy trataria algoritmos intercambiables; ninguno resolveria esta traduccion entre interfaces.